# Clase 04: Pilas abstractas, polimorfismo y pytest

Pregunta guía:

> Si todas son pilas, ¿por qué podríamos implementarlas de distintas formas?

Idea central:

> Una pila es un Tipo de Dato Abstracto definido por su comportamiento LIFO. La interfaz debe mantenerse estable, pero la implementación puede cambiar. Si varias clases implementan la misma interfaz, entonces deben poder usarse de manera polimórfica y pasar las mismas pruebas.

## Objetivos

Al terminar este notebook podrás:

- Explicar la relación entre TDA, interfaz y clase abstracta.
- Usar `ABC` y `abstractmethod` para definir un contrato.
- Comparar implementaciones de pila con `list`, `deque` y `tuple`.
- Explicar polimorfismo usando pilas.
- Escribir pruebas básicas con `assert`.
- Convertir pruebas a `pytest`.
- Usar `pytest.raises` y `pytest.mark.parametrize`.
- Entender que las pruebas verifican comportamiento, no detalles internos.

## Preparación

Esta celda localiza la carpeta de la clase e importa el código auxiliar.

In [1]:
from pathlib import Path
import sys
import subprocess

candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'clase_04']
raiz = next(
    candidato for candidato in candidatos
    if (candidato / 'src' / 'pilas.py').exists()
)

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.pilas import (
    PilaAbstracta,
    PilaLista,
    PilaDeque,
    PilaTupla,
    PilaNodo,
    vaciar_pila,
    transferir_elementos,
)

print(f'Material cargado desde: {raiz}')

Material cargado desde: c:\Users\0282193\Documents\GitHub\EstructurasDatos\clase_04


## Sección 1 — Recordatorio

Un **Tipo de Dato Abstracto** describe operaciones y comportamiento esperado sin fijar la implementación.

Una **interfaz** es el conjunto de operaciones públicas que el usuario puede llamar.

Una **implementación** es la forma concreta de guardar datos y ejecutar esas operaciones.

Una pila sigue la regla **LIFO**: último en entrar, primero en salir.

Una pila no es lo mismo que una lista. Una lista puede ser una herramienta interna para implementar una pila, pero la pila se define por su comportamiento.

### Preguntas

1. ¿Qué métodos necesita una pila? .pop(), .peek(), lista_vacía()
2. ¿Qué significa que `pop()` respete LIFO? es decir que el elemento al que eliminará, será el ultimo en añadirse
3. ¿Por qué decir que una pila *es una lista* puede ser confuso? por que tiene mas complejidad y también es para cosas mas especificas

## Sección 2 — Clase abstracta

Python permite definir contratos con `ABC` y `abstractmethod`.

- `ABC` indica que una clase puede tener métodos abstractos.
- `abstractmethod` marca métodos obligatorios para las subclases.
- Una clase abstracta no se instancia directamente.
- Si una subclase no implementa todo el contrato, Python lanza un `TypeError` al intentar instanciarla.

In [2]:
from abc import ABC, abstractmethod

class PilaAbstractaDemo(ABC):
    @abstractmethod
    def push(self, elemento):
        pass

    @abstractmethod
    def pop(self):
        pass

    @abstractmethod
    def peek(self):
        pass

    @abstractmethod
    def esta_vacia(self):
        pass

    @abstractmethod
    def tamano(self):
        pass

print('Contrato definido: PilaAbstractaDemo')

Contrato definido: PilaAbstractaDemo


### ¿Qué pasa si falta un método?

La siguiente clase intenta implementar solo una parte del contrato.

In [3]:
class PilaIncompleta(PilaAbstractaDemo):
    def push(self, elemento):
        pass

try:
    PilaIncompleta()
except TypeError as error:
    print('Python no permite instanciar la clase incompleta:')
    print(error)

Python no permite instanciar la clase incompleta:
Can't instantiate abstract class PilaIncompleta without an implementation for abstract methods 'esta_vacia', 'peek', 'pop', 'tamano'


### Ejercicio

Completa mentalmente el contrato de `PilaAbstracta` con estos métodos:

- `push(elemento)`
- `pop()`
- `peek()`
- `esta_vacia()`
- `tamano()`

Explica con tus palabras por qué `PilaAbstracta` no es una pila concreta.

PilaAbstracta no es una pila completa debido principalmente a que es abstracta solo da guia para otras cosas

## Sección 3 — Implementación `PilaLista`

`PilaLista` usa internamente una lista de Python.

Decisión de diseño: el final de la lista representa la cima de la pila.

Pseudocódigo:

```text
push(elemento):
    agregar elemento al final

pop():
    si está vacía, lanzar error
    quitar y regresar último elemento

peek():
    si está vacía, lanzar error
    regresar último elemento sin quitarlo
```

In [4]:
pila = PilaLista()
pila.push('A')
pila.push('B')

assert pila.tamano() == 2
assert pila.peek() == 'B'
assert pila.pop() == 'B'
assert pila.pop() == 'A'
assert pila.esta_vacia()

print('PilaLista pasa las pruebas rápidas.')

PilaLista pasa las pruebas rápidas.


### Ejercicio guiado

En tu copia del notebook, escribe el pseudocódigo de `PilaLista` con tus palabras. Después identifica qué operación de `list` corresponde a cada método de la pila.

list:[A,2,B]

push(20):
     agregar al final

list.pop():
    de esta forma quitamosel elemento que acabamos de agregar

list.peek():
    regresamos el elemento "B"

list.tamaño():
    nos regresa 3 siendo este el tamaño de la lista

list.esta_vacia():
    regresara false por que la lista tiene al menos un elemento




## Sección 4 — Implementación `PilaDeque`

`collections.deque` está diseñada para operaciones eficientes en los extremos. En esta implementación, la derecha del deque representa la cima.

Esta implementación prepara el camino para colas, donde también interesan los extremos.

In [ ]:
pila = PilaDeque()
for valor in [10, 20, 30]:
    pila.push(valor)

assert pila.peek() == 30
assert pila.pop() == 30
assert pila.pop() == 20
assert pila.tamano() == 1

print('PilaDeque pasa las pruebas rápidas.')

### Comparación rápida

- `PilaLista` suele ser la implementación más clara para empezar.
- `PilaDeque` es muy útil cuando queremos pensar en operaciones en ambos extremos.
- Ambas pueden respetar la misma interfaz.

## Sección 5 — Implementación `PilaTupla`

`tuple` es inmutable: no se modifica en sitio. Por eso, cada `push` o `pop` crea una nueva tupla interna.

Esta implementación no suele ser la opción normal para una pila con muchas operaciones, pero es muy útil para discutir inmutabilidad y decisiones de diseño.

In [ ]:
pila = PilaTupla()
pila.push('x')
pila.push('y')
pila.push('z')

assert pila.peek() == 'z'
assert pila.pop() == 'z'
assert pila.peek() == 'y'
assert pila.tamano() == 2

print('PilaTupla pasa las pruebas rápidas.')

### Preguntas

1. ¿Qué gana conceptualmente `PilaTupla`? que los elementos se quedan donde estan y no son cambiados ni modificados. 
2. ¿Qué costo paga cada vez que cambia? si la lista cambia de orden cambiara el ultimo elemento entonces cambiara el .pop(), .peek(). 
3. ¿La interfaz pública cambió respecto a `PilaLista`? si cambia, justo por que no puedes cambiar las cosas cambia.

## Sección 6 — Reto opcional: `PilaNodo`

`PilaNodo` usa nodos y referencias. Esta parte es opcional porque listas ligadas se estudiarán después.

Idea:

```text
cima -> Nodo(valor='C') -> Nodo(valor='B') -> Nodo(valor='A') -> None
```

El reto consiste en pensar cómo `push` cambia la cima y cómo `pop` avanza al siguiente nodo.

In [ ]:
pila = PilaNodo()
pila.push('uno')
pila.push('dos')

assert pila.peek() == 'dos'
assert pila.pop() == 'dos'
assert pila.pop() == 'uno'
assert pila.esta_vacia()

print('Reto PilaNodo: implementación disponible para explorar.')

## Sección 7 — Comparación de implementaciones

| Implementación | Herramienta interna | Ventaja | Desventaja | Cuándo podría tener sentido |
| --- | --- | --- | --- | --- |
| `PilaLista` | `list` | Simple y natural en Python | No muestra nodos ni referencias | Primera implementación del proyecto |
| `PilaDeque` | `deque` | Extremos eficientes | Oculta detalles internos | Puente hacia colas |
| `PilaTupla` | `tuple` | Discute inmutabilidad | Crea una nueva tupla al cambiar | Ejemplo conceptual |
| `PilaNodo` | nodos | Prepara listas ligadas | Más código y referencias | Reto o tema futuro |

### Reflexión

1. ¿Cuál implementación fue más clara? PilaLista
2. ¿Cuál parece más eficiente para muchas operaciones? PilaTupla
3. ¿Cuál oculta más detalles? PilaDeque
4. ¿Cuál ayuda más a entender estructuras enlazadas? PilaNodo

## Sección 8 — Polimorfismo

Polimorfismo significa que la misma función puede trabajar con objetos distintos si respetan la misma interfaz.

La función no necesita saber cómo está implementada la pila. Solo necesita que el objeto tenga los métodos esperados.

In [5]:
def preparar_pila(ClasePila):
    pila = ClasePila()
    for elemento in ['A', 'B', 'C']:
        pila.push(elemento)
    return pila

for ClasePila in [PilaLista, PilaDeque, PilaTupla]:
    pila = preparar_pila(ClasePila)
    print(ClasePila.__name__, vaciar_pila(pila))

PilaLista ['C', 'B', 'A']
PilaDeque ['C', 'B', 'A']
PilaTupla ['C', 'B', 'A']


In [6]:
origen = PilaLista()
destino = PilaDeque()

for numero in [1, 2, 3]:
    origen.push(numero)

transferir_elementos(origen, destino)

assert origen.esta_vacia()
assert destino.pop() == 1
assert destino.pop() == 2
assert destino.pop() == 3
print('Transferencia entre implementaciones distintas completada.')

Transferencia entre implementaciones distintas completada.


### Ejercicio

Escribe una función `copiar_a_lista(pila)` que reciba cualquier pila y regrese una lista con los elementos en orden de salida. Puedes inspirarte en `vaciar_pila`, pero piensa si tu función debería modificar o no modificar la pila.

In [ ]:
def copiar_a_lista(Pila):
    pila = Pila()
    for elemento in [pila]:
        pila.push(elemento)
        pila.pop(elemento)
    return pila



## Sección 9 — Pruebas con `assert`

`assert` verifica una expectativa. Si la condición es falsa, Python detiene la ejecución con un `AssertionError`.

Una prueba debe tener una expectativa clara.

In [ ]:
pila = PilaLista()
assert pila.esta_vacia()

pila.push(100)
assert not pila.esta_vacia()
assert pila.tamano() == 1
assert pila.peek() == 100
assert pila.pop() == 100
assert pila.esta_vacia()

print('Pruebas con assert completadas.')

### Ejercicios con `assert`

Escribe o modifica pruebas para verificar:

push: Inserta un elemento en la cima.

pop: Elimina el último elemento agregado. Si la pila está vacía, puede lanzar un error.

peek: Consulta el último elemento sin eliminarlo; en una pila vacía, debe manejarse con precaución.

isEmpty: Verifica si la pila está vacía.

## Sección 10 — Introducción a pytest

`pytest` permite organizar pruebas en archivos y ejecutarlas desde terminal.

Convenciones básicas:

- Carpeta: `tests/`
- Archivo: `tests/test_pilas.py`
- Funciones de prueba: empiezan con `test_`

Para ejecutar desde la carpeta `clase_04/`:

```bash
pytest
```

In [ ]:
test_path = raiz / 'tests' / 'test_pilas.py'
print(test_path)
print(test_path.read_text(encoding='utf-8').splitlines()[:12])

### Celda de notebook contra archivo de pruebas

Una celda del notebook sirve para experimentar y explicar.

Un archivo de pruebas sirve para repetir automáticamente verificaciones y detectar regresiones.

## Sección 11 — `pytest.raises`

Algunas pruebas verifican que ocurra un error esperado. Para pila vacía, acordamos que `pop()` y `peek()` deben lanzar `IndexError`.

In [3]:
import pytest

pila = PilaLista()
with pytest.raises(IndexError):
    pila.pop()

with pytest.raises(IndexError):
    pila.peek()

print('Errores esperados verificados con pytest.raises.')

ModuleNotFoundError: No module named 'pytest'

### Ejercicio

Agrega una prueba equivalente para `PilaDeque` o `PilaTupla`.

In [6]:
import pytest

pila = PilaDeque()
with pytest.raises(IndexError):
    pila.pop()

with pytest.raises(IndexError):
    pila.peek()

print("Errores esperados verificados con pytest.raises en PilaDeque.")


ModuleNotFoundError: No module named 'pytest'

In [5]:
import pytest

pila = PilaTupla()
with pytest.raises(IndexError):
    pila.pop()

with pytest.raises(IndexError):
    pila.peek()

print("Errores esperados verificados con pytest.raises en PilaTupla.")


ModuleNotFoundError: No module named 'pytest'

## Sección 12 — `pytest.mark.parametrize`

`parametrize` sirve para aplicar la misma prueba a varios casos. Aquí es muy útil porque tenemos varias implementaciones del mismo TDA.

Idea:

```python
@pytest.mark.parametrize('ClasePila', [PilaLista, PilaDeque, PilaTupla])
def test_pop_respeta_lifo(ClasePila):
    pila = ClasePila()
    ...
```

In [4]:
for ClasePila in [PilaLista, PilaDeque, PilaTupla]:
    pila = ClasePila()
    pila.push('primero')
    pila.push('segundo')
    assert pila.pop() == 'segundo'
    assert pila.pop() == 'primero'

print('La misma prueba conceptual funciona para tres implementaciones.')

NameError: name 'PilaLista' is not defined

### Ejecutar pytest desde el notebook

La forma normal será ejecutar `pytest` en terminal. Esta celda lo ejecuta para confirmar que el entorno está listo.

In [ ]:
resultado = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=raiz,
    text=True,
    capture_output=True,
)
print(resultado.stdout)
if resultado.stderr:
    print(resultado.stderr)
assert resultado.returncode == 0

## Sección 13 — Cierre

Preguntas finales:

1. ¿Qué ganamos con una interfaz común? mas vesatilidad
2. ¿Qué ganamos con pruebas comunes? mas comodidad y facilidad 
3. ¿Qué implementación debería entrar al proyecto principal? la de PilaLista
4. ¿Qué otras implementaciones podríamos imaginar? Coomo  el ctrl z en las cpu
5. ¿Qué prueba no debería depender de detalles internos? la PilaNodo

## Entrega

Cada estudiante debe crear:

```text
entregas/clase_04/nombre/
├── clase_04_pilas_nombre.ipynb
├── resumen.md
└── evidencia_pytest.txt
```

`resumen.md` debe incluir:

- cuál implementación fue más clara;
- cuál implementación parece más eficiente;
- qué aprendiste de `pytest`;
- qué prueba te pareció más importante;
- qué duda te queda.